In [ ]:
# Parameters and dev-only write guard.
import csv
import os
import re

# The production switch remains a separate reviewed change, as on c3db0a32.
dbutils.widgets.text("prefix", "openalex_dev.rohan_lab.fr1006_")
dbutils.widgets.dropdown("mode", "daily", ["daily", "seed"])
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("max_mints", "200")
prefix = dbutils.widgets.get("prefix").strip()
mode = dbutils.widgets.get("mode")
run_id = dbutils.widgets.get("run_id").strip()
assert prefix == "openalex_dev.rohan_lab.fr1006_", "v0 writes only to openalex_dev (switch to prod is a separate, reviewed change)"
assert run_id, "run_id parameter required ({{job.run_id}})"
assert mode in ("daily", "seed"), "mode must be daily or seed"
if mode == "daily":
    int(dbutils.widgets.get("max_mints"))
print({"prefix": prefix, "mode": mode, "run_id": "fd" + run_id,
       "max_mints": dbutils.widgets.get("max_mints")})


In [ ]:
%sql
-- guard_ror
BEGIN
IF :mode = 'daily' THEN
SELECT assert_true(
  (SELECT count(*) FROM openalex.institutions.ror) >= 100000
  AND (SELECT count_if(coalesce(array_contains(types, 'funder'), false)) FROM openalex.institutions.ror) >= 15000
  AND (SELECT max(updated_date) FROM openalex.institutions.ror) IS NOT NULL
  AND (SELECT count(*) FROM (SELECT DISTINCT lower(regexp_extract(id, '([0-9a-z]{9})/*$', 1)) rid FROM openalex.institutions.ror) c
       LEFT ANTI JOIN (SELECT DISTINCT lower(regexp_extract(id, '([0-9a-z]{9})/*$', 1)) rid FROM openalex.institutions.ror_raw) h ON h.rid = c.rid) = 0,
  'ROR copy too small, undated, or history incomplete; nothing written (fail closed)') ok;
END IF;
END;


In [ ]:
%sql
-- Pin the pre-run funder version or the current seed version before any registry write.
BEGIN
DECLARE pinned_version BIGINT;
DECLARE statement_text STRING;
IF :mode = 'daily' THEN
  SET statement_text = 'SELECT max(version) FROM (DESCRIBE HISTORY ' || trim(:prefix) || 'funder)';
  EXECUTE IMMEDIATE statement_text INTO pinned_version;
  -- IDENTIFIER + VERSION AS OF is unsupported in Unity Catalog; bind a literal relation in this temporary view.
  SET statement_text = 'CREATE OR REPLACE TEMP VIEW fr1006_prev_funder AS SELECT * FROM ' || trim(:prefix) || 'funder VERSION AS OF ' || pinned_version;
ELSE
  SET pinned_version = (SELECT max(version) FROM (DESCRIBE HISTORY openalex.funders.funders));
  SET statement_text = 'CREATE OR REPLACE TEMP VIEW fr1006_seed_funders AS SELECT * FROM openalex.funders.funders VERSION AS OF ' || pinned_version;
END IF;
EXECUTE IMMEDIATE statement_text;
SELECT :mode mode, ('fd' || trim(:run_id)) run_id, pinned_version;
END;


In [ ]:
%sql
-- create_funder
BEGIN
IF :mode = 'seed' THEN
CREATE TABLE IDENTIFIER(trim(:prefix) || 'funder') (
  funder_id BIGINT NOT NULL, status STRING NOT NULL, merged_into_id BIGINT, ror_id STRING, ror_primary BOOLEAN, parent_id BIGINT,
  display_name STRING, alt_names ARRAY<STRING>, country_code STRING, homepage_url STRING, wikidata_id STRING, description STRING,
  image_url STRING, image_thumbnail_url STRING, created_date TIMESTAMP, updated_date TIMESTAMP, created_by STRING, run_id STRING,
  mint_key STRING) USING DELTA;
END IF;
END;


In [ ]:
%sql
-- create_funder_id
BEGIN
IF :mode = 'seed' THEN
CREATE TABLE IDENTIFIER(trim(:prefix) || 'funder_id') (
  id_type STRING NOT NULL, id_value STRING NOT NULL, funder_id BIGINT NOT NULL, origin_funder_id BIGINT, is_primary BOOLEAN,
  source STRING, added_at TIMESTAMP, run_id STRING) USING DELTA;
END IF;
END;


In [ ]:
%sql
-- create_source_record
BEGIN
IF :mode = 'seed' THEN
CREATE TABLE IDENTIFIER(trim(:prefix) || 'funder_source_record') (
  source STRING NOT NULL, source_key STRING NOT NULL, name STRING, alt_names ARRAY<STRING>, country STRING, ror_id STRING,
  fundref_ids ARRAY<STRING>, replaced_by ARRAY<STRING>, record_status STRING, is_funder BOOLEAN, homepage STRING, record_json STRING,
  fingerprint STRING, first_seen_at TIMESTAMP, fetched_at TIMESTAMP, changed_run_id STRING,
  outcome STRING, reason STRING, funder_id BIGINT, decided_run_id STRING) USING DELTA;
END IF;
END;


In [ ]:
%sql
-- create_decision
BEGIN
IF :mode = 'seed' THEN
CREATE TABLE IDENTIFIER(trim(:prefix) || 'funder_decision') (
  run_id STRING NOT NULL, decided_at TIMESTAMP, source STRING, source_key STRING, action STRING NOT NULL, funder_id BIGINT,
  ror_rid STRING, fundref STRING, display_name STRING, country STRING, evidence STRING) USING DELTA;
END IF;
END;


In [ ]:
%sql
-- seed_decisions
BEGIN
IF :mode = 'seed' THEN
INSERT INTO IDENTIFIER(trim(:prefix) || 'funder_decision')
WITH v AS (SELECT * FROM fr1006_seed_funders),
r AS (
  SELECT a.funder_id,
    CASE WHEN a.merge_into_id IS NULL THEN NULL WHEN b.merge_into_id IS NULL THEN a.merge_into_id
         WHEN c.merge_into_id IS NULL THEN b.merge_into_id WHEN d.merge_into_id IS NULL THEN c.merge_into_id END root
  FROM v a LEFT JOIN v b ON b.funder_id = a.merge_into_id LEFT JOIN v c ON c.funder_id = b.merge_into_id
  LEFT JOIN v d ON d.funder_id = c.merge_into_id
),
rorx AS (
  SELECT lower(regexp_extract(id, '([0-9a-z]{9})/*$', 1)) rid,
    array_distinct(filter(transform(flatten(transform(filter(coalesce(external_ids, array()), e -> lower(e.type) = 'fundref'),
      e -> coalesce(e.all, array()))), x -> regexp_extract(trim(x), '^(?:(?:https?://)?(?:dx\\.)?(?:doi\\.org/)?10\\.13039/)?([0-9]{5,})$', 1)), x -> x <> '')) fundrefs,
    array_distinct(filter(transform(coalesce(transform(filter(coalesce(names, array()), n -> NOT array_contains(n.types, 'acronym')), n -> n.value), array()), x -> regexp_replace(trim(regexp_replace(lower(x), '[^\\p{L}\\p{N}]+', ' ')), '^the ', '')), x -> length(x) >= 4)) nk
  FROM openalex.institutions.ror
),
live AS (
  SELECT v.funder_id, CAST(v.crossref_id AS STRING) fr, lower(regexp_extract(v.ror_id, '([0-9a-z]{9})/*$', 1)) rid, regexp_replace(trim(regexp_replace(lower(v.display_name), '[^\\p{L}\\p{N}]+', ' ')), '^the ', '') dn, coalesce(w.works_count, 0) works,
    array_distinct(filter(transform(coalesce(concat(array(v.display_name), coalesce(from_json(v.alternate_titles, 'array<string>'), array())), array()), x -> regexp_replace(trim(regexp_replace(lower(x), '[^\\p{L}\\p{N}]+', ' ')), '^the ', '')), x -> length(x) >= 4)) nk
  FROM v LEFT JOIN (SELECT id, works_count FROM openalex.funders.funders_api) w ON w.id = v.funder_id
  WHERE v.merge_into_id IS NULL AND nullif(v.ror_id, '') IS NOT NULL
),
shared AS (SELECT rid FROM live GROUP BY rid HAVING count(*) > 1),
mem AS (
  SELECT l.*, coalesce(true AND array_contains(x.fundrefs, l.fr), false) in_xw, coalesce(array_contains(x.nk, l.dn), false) exact,
    coalesce(arrays_overlap(l.nk, x.nk), false) name_ok
  FROM live l JOIN shared s ON s.rid = l.rid LEFT JOIN rorx x ON x.rid = l.rid
),
tier AS (SELECT *, CASE WHEN in_xw AND exact THEN 1 WHEN in_xw AND name_ok THEN 2 WHEN in_xw THEN 3 WHEN exact THEN 4
                        WHEN name_ok THEN 5 ELSE 9 END t FROM mem),
best AS (SELECT rid, min(t) bt FROM tier GROUP BY rid),
keep AS (SELECT t.rid, max_by(t.funder_id, t.works * 10000000000 - t.funder_id) keeper, count(*) n_at, max(b.bt) bt
         FROM tier t JOIN best b ON b.rid = t.rid AND t.t = b.bt WHERE b.bt < 9 GROUP BY t.rid),
shared_dec AS (
  SELECT m.funder_id, m.rid, m.t, k.keeper, coalesce(k.bt, 9) bt,
    CASE WHEN (k.n_at = 1 OR k.bt IN (1, 4)) AND m.funder_id = k.keeper THEN 'PRIMARY_SHARED_ROR'
         WHEN k.n_at = 1 OR k.bt IN (1, 4) THEN 'SECONDARY_SHARED_ROR' ELSE 'HOLD_SHARED_ROR_TIE' END d,
    coalesce((k.n_at = 1 OR k.bt IN (1, 4)) AND m.funder_id <> k.keeper AND m.exact, false) merge_candidate
  FROM tier m LEFT JOIN keep k ON k.rid = m.rid
)
SELECT ('fd' || trim(:run_id)), current_timestamp(), 'openalex_v41', CAST(s.funder_id AS STRING), s.d, s.funder_id, s.rid, v.crossref_id,
  v.display_name, v.country_code,
  to_json(named_struct('tier', s.t, 'best_tier', s.bt, 'keeper', s.keeper, 'merge_candidate', s.merge_candidate, 'works', l.works))
FROM shared_dec s JOIN v ON v.funder_id = s.funder_id JOIN live l ON l.funder_id = s.funder_id;
END IF;
END;


In [ ]:
%sql
-- seed_funder
BEGIN
IF :mode = 'seed' THEN
INSERT INTO IDENTIFIER(trim(:prefix) || 'funder')
WITH v AS (SELECT * FROM fr1006_seed_funders),
r AS (
  SELECT a.funder_id,
    CASE WHEN a.merge_into_id IS NULL THEN NULL WHEN b.merge_into_id IS NULL THEN a.merge_into_id
         WHEN c.merge_into_id IS NULL THEN b.merge_into_id WHEN d.merge_into_id IS NULL THEN c.merge_into_id END root
  FROM v a LEFT JOIN v b ON b.funder_id = a.merge_into_id LEFT JOIN v c ON c.funder_id = b.merge_into_id
  LEFT JOIN v d ON d.funder_id = c.merge_into_id
),
rorx AS (
  SELECT lower(regexp_extract(id, '([0-9a-z]{9})/*$', 1)) rid,
    array_distinct(filter(transform(flatten(transform(filter(coalesce(external_ids, array()), e -> lower(e.type) = 'fundref'),
      e -> coalesce(e.all, array()))), x -> regexp_extract(trim(x), '^(?:(?:https?://)?(?:dx\\.)?(?:doi\\.org/)?10\\.13039/)?([0-9]{5,})$', 1)), x -> x <> '')) fundrefs,
    array_distinct(filter(transform(coalesce(transform(filter(coalesce(names, array()), n -> NOT array_contains(n.types, 'acronym')), n -> n.value), array()), x -> regexp_replace(trim(regexp_replace(lower(x), '[^\\p{L}\\p{N}]+', ' ')), '^the ', '')), x -> length(x) >= 4)) nk
  FROM openalex.institutions.ror
),
live AS (
  SELECT v.funder_id, CAST(v.crossref_id AS STRING) fr, lower(regexp_extract(v.ror_id, '([0-9a-z]{9})/*$', 1)) rid, regexp_replace(trim(regexp_replace(lower(v.display_name), '[^\\p{L}\\p{N}]+', ' ')), '^the ', '') dn, coalesce(w.works_count, 0) works,
    array_distinct(filter(transform(coalesce(concat(array(v.display_name), coalesce(from_json(v.alternate_titles, 'array<string>'), array())), array()), x -> regexp_replace(trim(regexp_replace(lower(x), '[^\\p{L}\\p{N}]+', ' ')), '^the ', '')), x -> length(x) >= 4)) nk
  FROM v LEFT JOIN (SELECT id, works_count FROM openalex.funders.funders_api) w ON w.id = v.funder_id
  WHERE v.merge_into_id IS NULL AND nullif(v.ror_id, '') IS NOT NULL
),
shared AS (SELECT rid FROM live GROUP BY rid HAVING count(*) > 1),
mem AS (
  SELECT l.*, coalesce(true AND array_contains(x.fundrefs, l.fr), false) in_xw, coalesce(array_contains(x.nk, l.dn), false) exact,
    coalesce(arrays_overlap(l.nk, x.nk), false) name_ok
  FROM live l JOIN shared s ON s.rid = l.rid LEFT JOIN rorx x ON x.rid = l.rid
),
tier AS (SELECT *, CASE WHEN in_xw AND exact THEN 1 WHEN in_xw AND name_ok THEN 2 WHEN in_xw THEN 3 WHEN exact THEN 4
                        WHEN name_ok THEN 5 ELSE 9 END t FROM mem),
best AS (SELECT rid, min(t) bt FROM tier GROUP BY rid),
keep AS (SELECT t.rid, max_by(t.funder_id, t.works * 10000000000 - t.funder_id) keeper, count(*) n_at, max(b.bt) bt
         FROM tier t JOIN best b ON b.rid = t.rid AND t.t = b.bt WHERE b.bt < 9 GROUP BY t.rid),
shared_dec AS (
  SELECT m.funder_id, m.rid, m.t, k.keeper, coalesce(k.bt, 9) bt,
    CASE WHEN (k.n_at = 1 OR k.bt IN (1, 4)) AND m.funder_id = k.keeper THEN 'PRIMARY_SHARED_ROR'
         WHEN k.n_at = 1 OR k.bt IN (1, 4) THEN 'SECONDARY_SHARED_ROR' ELSE 'HOLD_SHARED_ROR_TIE' END d,
    coalesce((k.n_at = 1 OR k.bt IN (1, 4)) AND m.funder_id <> k.keeper AND m.exact, false) merge_candidate
  FROM tier m LEFT JOIN keep k ON k.rid = m.rid
)
SELECT v.funder_id, CASE WHEN v.merge_into_id IS NULL THEN 'active' ELSE 'merged' END, r.root,
  nullif(v.ror_id, ''),   -- every funder keeps the ROR it has; only the primary one resolves the ROR id
  CASE WHEN nullif(v.ror_id, '') IS NULL OR v.merge_into_id IS NOT NULL THEN NULL WHEN s.d IN ('SECONDARY_SHARED_ROR', 'HOLD_SHARED_ROR_TIE') THEN false ELSE true END,
  CAST(NULL AS BIGINT),
  v.display_name, from_json(v.alternate_titles, 'array<string>'), v.country_code, v.homepage_url, v.wikidata_id, v.description,
  v.image_url, v.image_thumbnail_url, coalesce(try_cast(substr(v.created_date, 1, 19) AS TIMESTAMP), v.updated_date), v.updated_date,
  'seed_v41', ('fd' || trim(:run_id)), CAST(NULL AS STRING)
FROM v JOIN r ON r.funder_id = v.funder_id LEFT JOIN shared_dec s ON s.funder_id = v.funder_id
UNION ALL
SELECT funder_id, 'withdrawn', CAST(NULL AS BIGINT), nullif(ror_id, ''), CAST(NULL AS BOOLEAN), CAST(NULL AS BIGINT), get_json_object(before_image_json, '$.display_name'),
  from_json(get_json_object(before_image_json, '$.alternate_titles'), 'array<string>'), get_json_object(before_image_json, '$.country_code'), get_json_object(before_image_json, '$.homepage_url'), get_json_object(before_image_json, '$.wikidata_id'),
  get_json_object(before_image_json, '$.description'), get_json_object(before_image_json, '$.image_url'), get_json_object(before_image_json, '$.image_thumbnail_url'), try_cast(substr(get_json_object(before_image_json, '$.created_date'), 1, 19) AS TIMESTAMP),
  deleted_at, 'seed_deleted', ('fd' || trim(:run_id)), CAST(NULL AS STRING)
FROM openalex.funders.deleted_funders WHERE funder_id NOT IN (SELECT funder_id FROM fr1006_seed_funders);
END IF;
END;


In [ ]:
%sql
-- seed_funder_id
BEGIN
IF :mode = 'seed' THEN
INSERT INTO IDENTIFIER(trim(:prefix) || 'funder_id')
WITH v AS (SELECT * FROM fr1006_seed_funders),
r AS (
  SELECT a.funder_id,
    CASE WHEN a.merge_into_id IS NULL THEN NULL WHEN b.merge_into_id IS NULL THEN a.merge_into_id
         WHEN c.merge_into_id IS NULL THEN b.merge_into_id WHEN d.merge_into_id IS NULL THEN c.merge_into_id END root
  FROM v a LEFT JOIN v b ON b.funder_id = a.merge_into_id LEFT JOIN v c ON c.funder_id = b.merge_into_id
  LEFT JOIN v d ON d.funder_id = c.merge_into_id
),
rorx AS (
  SELECT lower(regexp_extract(id, '([0-9a-z]{9})/*$', 1)) rid,
    array_distinct(filter(transform(flatten(transform(filter(coalesce(external_ids, array()), e -> lower(e.type) = 'fundref'),
      e -> coalesce(e.all, array()))), x -> regexp_extract(trim(x), '^(?:(?:https?://)?(?:dx\\.)?(?:doi\\.org/)?10\\.13039/)?([0-9]{5,})$', 1)), x -> x <> '')) fundrefs,
    array_distinct(filter(transform(coalesce(transform(filter(coalesce(names, array()), n -> NOT array_contains(n.types, 'acronym')), n -> n.value), array()), x -> regexp_replace(trim(regexp_replace(lower(x), '[^\\p{L}\\p{N}]+', ' ')), '^the ', '')), x -> length(x) >= 4)) nk
  FROM openalex.institutions.ror
),
live AS (
  SELECT v.funder_id, CAST(v.crossref_id AS STRING) fr, lower(regexp_extract(v.ror_id, '([0-9a-z]{9})/*$', 1)) rid, regexp_replace(trim(regexp_replace(lower(v.display_name), '[^\\p{L}\\p{N}]+', ' ')), '^the ', '') dn, coalesce(w.works_count, 0) works,
    array_distinct(filter(transform(coalesce(concat(array(v.display_name), coalesce(from_json(v.alternate_titles, 'array<string>'), array())), array()), x -> regexp_replace(trim(regexp_replace(lower(x), '[^\\p{L}\\p{N}]+', ' ')), '^the ', '')), x -> length(x) >= 4)) nk
  FROM v LEFT JOIN (SELECT id, works_count FROM openalex.funders.funders_api) w ON w.id = v.funder_id
  WHERE v.merge_into_id IS NULL AND nullif(v.ror_id, '') IS NOT NULL
),
shared AS (SELECT rid FROM live GROUP BY rid HAVING count(*) > 1),
mem AS (
  SELECT l.*, coalesce(true AND array_contains(x.fundrefs, l.fr), false) in_xw, coalesce(array_contains(x.nk, l.dn), false) exact,
    coalesce(arrays_overlap(l.nk, x.nk), false) name_ok
  FROM live l JOIN shared s ON s.rid = l.rid LEFT JOIN rorx x ON x.rid = l.rid
),
tier AS (SELECT *, CASE WHEN in_xw AND exact THEN 1 WHEN in_xw AND name_ok THEN 2 WHEN in_xw THEN 3 WHEN exact THEN 4
                        WHEN name_ok THEN 5 ELSE 9 END t FROM mem),
best AS (SELECT rid, min(t) bt FROM tier GROUP BY rid),
keep AS (SELECT t.rid, max_by(t.funder_id, t.works * 10000000000 - t.funder_id) keeper, count(*) n_at, max(b.bt) bt
         FROM tier t JOIN best b ON b.rid = t.rid AND t.t = b.bt WHERE b.bt < 9 GROUP BY t.rid),
shared_dec AS (
  SELECT m.funder_id, m.rid, m.t, k.keeper, coalesce(k.bt, 9) bt,
    CASE WHEN (k.n_at = 1 OR k.bt IN (1, 4)) AND m.funder_id = k.keeper THEN 'PRIMARY_SHARED_ROR'
         WHEN k.n_at = 1 OR k.bt IN (1, 4) THEN 'SECONDARY_SHARED_ROR' ELSE 'HOLD_SHARED_ROR_TIE' END d,
    coalesce((k.n_at = 1 OR k.bt IN (1, 4)) AND m.funder_id <> k.keeper AND m.exact, false) merge_candidate
  FROM tier m LEFT JOIN keep k ON k.rid = m.rid
)
SELECT 'fundref', CAST(v.crossref_id AS STRING), coalesce(r.root, v.funder_id), v.funder_id, true, 'seed_v41', current_timestamp(), ('fd' || trim(:run_id))
FROM v JOIN r ON r.funder_id = v.funder_id WHERE v.crossref_id IS NOT NULL AND true
UNION ALL
SELECT 'fundref', CAST(d.crossref_id AS STRING), d.funder_id, d.funder_id, true, 'seed_deleted', current_timestamp(), ('fd' || trim(:run_id))
FROM openalex.funders.deleted_funders d LEFT ANTI JOIN v ON CAST(v.crossref_id AS STRING) = CAST(d.crossref_id AS STRING)
WHERE true AND d.crossref_id IS NOT NULL AND d.funder_id NOT IN (SELECT funder_id FROM v)
UNION ALL
SELECT 'ror', lower(regexp_extract(f.ror_id, '([0-9a-z]{9})/*$', 1)), f.funder_id, f.funder_id, true, 'seed_v41', current_timestamp(), ('fd' || trim(:run_id))
FROM IDENTIFIER(trim(:prefix) || 'funder') f WHERE f.status = 'active' AND f.ror_primary
UNION ALL   -- a merge loser's own ROR id points at its winner, unless an active funder already holds it
SELECT 'ror', x.rid, max(x.root), max(x.funder_id), false, 'seed_v41_merged', current_timestamp(), ('fd' || trim(:run_id))
FROM (SELECT lower(regexp_extract(v.ror_id, '([0-9a-z]{9})/*$', 1)) rid, r.root, v.funder_id FROM v JOIN r ON r.funder_id = v.funder_id
      WHERE v.merge_into_id IS NOT NULL AND nullif(v.ror_id, '') IS NOT NULL) x
LEFT ANTI JOIN (SELECT lower(regexp_extract(ror_id, '([0-9a-z]{9})/*$', 1)) rid FROM IDENTIFIER(trim(:prefix) || 'funder') WHERE status = 'active' AND ror_id IS NOT NULL) a ON a.rid = x.rid   -- held by an active funder, primary or not
GROUP BY x.rid HAVING count(DISTINCT x.root) = 1;
END IF;
END;


In [ ]:
%sql
-- seed_source_v41
BEGIN
IF :mode = 'seed' THEN
INSERT INTO IDENTIFIER(trim(:prefix) || 'funder_source_record')
WITH v AS (SELECT * FROM fr1006_seed_funders),
r AS (
  SELECT a.funder_id,
    CASE WHEN a.merge_into_id IS NULL THEN NULL WHEN b.merge_into_id IS NULL THEN a.merge_into_id
         WHEN c.merge_into_id IS NULL THEN b.merge_into_id WHEN d.merge_into_id IS NULL THEN c.merge_into_id END root
  FROM v a LEFT JOIN v b ON b.funder_id = a.merge_into_id LEFT JOIN v c ON c.funder_id = b.merge_into_id
  LEFT JOIN v d ON d.funder_id = c.merge_into_id
),
rorx AS (
  SELECT lower(regexp_extract(id, '([0-9a-z]{9})/*$', 1)) rid,
    array_distinct(filter(transform(flatten(transform(filter(coalesce(external_ids, array()), e -> lower(e.type) = 'fundref'),
      e -> coalesce(e.all, array()))), x -> regexp_extract(trim(x), '^(?:(?:https?://)?(?:dx\\.)?(?:doi\\.org/)?10\\.13039/)?([0-9]{5,})$', 1)), x -> x <> '')) fundrefs,
    array_distinct(filter(transform(coalesce(transform(filter(coalesce(names, array()), n -> NOT array_contains(n.types, 'acronym')), n -> n.value), array()), x -> regexp_replace(trim(regexp_replace(lower(x), '[^\\p{L}\\p{N}]+', ' ')), '^the ', '')), x -> length(x) >= 4)) nk
  FROM openalex.institutions.ror
),
live AS (
  SELECT v.funder_id, CAST(v.crossref_id AS STRING) fr, lower(regexp_extract(v.ror_id, '([0-9a-z]{9})/*$', 1)) rid, regexp_replace(trim(regexp_replace(lower(v.display_name), '[^\\p{L}\\p{N}]+', ' ')), '^the ', '') dn, coalesce(w.works_count, 0) works,
    array_distinct(filter(transform(coalesce(concat(array(v.display_name), coalesce(from_json(v.alternate_titles, 'array<string>'), array())), array()), x -> regexp_replace(trim(regexp_replace(lower(x), '[^\\p{L}\\p{N}]+', ' ')), '^the ', '')), x -> length(x) >= 4)) nk
  FROM v LEFT JOIN (SELECT id, works_count FROM openalex.funders.funders_api) w ON w.id = v.funder_id
  WHERE v.merge_into_id IS NULL AND nullif(v.ror_id, '') IS NOT NULL
),
shared AS (SELECT rid FROM live GROUP BY rid HAVING count(*) > 1),
mem AS (
  SELECT l.*, coalesce(true AND array_contains(x.fundrefs, l.fr), false) in_xw, coalesce(array_contains(x.nk, l.dn), false) exact,
    coalesce(arrays_overlap(l.nk, x.nk), false) name_ok
  FROM live l JOIN shared s ON s.rid = l.rid LEFT JOIN rorx x ON x.rid = l.rid
),
tier AS (SELECT *, CASE WHEN in_xw AND exact THEN 1 WHEN in_xw AND name_ok THEN 2 WHEN in_xw THEN 3 WHEN exact THEN 4
                        WHEN name_ok THEN 5 ELSE 9 END t FROM mem),
best AS (SELECT rid, min(t) bt FROM tier GROUP BY rid),
keep AS (SELECT t.rid, max_by(t.funder_id, t.works * 10000000000 - t.funder_id) keeper, count(*) n_at, max(b.bt) bt
         FROM tier t JOIN best b ON b.rid = t.rid AND t.t = b.bt WHERE b.bt < 9 GROUP BY t.rid),
shared_dec AS (
  SELECT m.funder_id, m.rid, m.t, k.keeper, coalesce(k.bt, 9) bt,
    CASE WHEN (k.n_at = 1 OR k.bt IN (1, 4)) AND m.funder_id = k.keeper THEN 'PRIMARY_SHARED_ROR'
         WHEN k.n_at = 1 OR k.bt IN (1, 4) THEN 'SECONDARY_SHARED_ROR' ELSE 'HOLD_SHARED_ROR_TIE' END d,
    coalesce((k.n_at = 1 OR k.bt IN (1, 4)) AND m.funder_id <> k.keeper AND m.exact, false) merge_candidate
  FROM tier m LEFT JOIN keep k ON k.rid = m.rid
)
SELECT 'openalex_v41', CAST(v.funder_id AS STRING), v.display_name, from_json(v.alternate_titles, 'array<string>'), v.country_code,
  nullif(v.ror_id, ''), CASE WHEN v.crossref_id IS NOT NULL THEN array(CAST(v.crossref_id AS STRING)) END,
  CASE WHEN v.merge_into_id IS NOT NULL THEN array(CAST(v.merge_into_id AS STRING)) END,
  CASE WHEN v.merge_into_id IS NULL THEN 'active' ELSE 'merged' END, true, v.homepage_url,
  to_json(struct(v.*), map('timestampFormat', "yyyy-MM-dd'T'HH:mm:ss.SSSSSSXXX")),   -- microseconds, so the compat passthrough reproduces v41 exactly
  sha2(to_json(struct(v.*)), 256), current_timestamp(), current_timestamp(), ('fd' || trim(:run_id)),
  CASE WHEN s.d = 'HOLD_SHARED_ROR_TIE' THEN 'held' ELSE 'accepted' END,
  CASE WHEN s.d IS NOT NULL THEN s.d || CASE WHEN s.merge_candidate THEN ' (MERGE_CANDIDATE into F' || s.keeper || ')' ELSE '' END END,
  coalesce(r.root, v.funder_id), ('fd' || trim(:run_id))
FROM v JOIN r ON r.funder_id = v.funder_id LEFT JOIN shared_dec s ON s.funder_id = v.funder_id;
END IF;
END;


In [ ]:
%sql
-- create_ever_issued
BEGIN
IF :mode = 'seed' THEN
CREATE TABLE IDENTIFIER(trim(:prefix) || 'f_ids_ever_issued') USING DELTA AS
SELECT funder_id, 'live_v41' source, try_cast(substr(created_date, 1, 19) AS TIMESTAMP) issued_at FROM fr1006_seed_funders
UNION ALL
SELECT funder_id, 'deleted', deleted_at FROM openalex.funders.deleted_funders WHERE funder_id NOT IN (SELECT funder_id FROM fr1006_seed_funders)
UNION ALL
SELECT funder_id, created_by, created_date FROM IDENTIFIER(trim(:prefix) || 'funder') WHERE created_by LIKE 'mint%';
END IF;
END;


In [ ]:
# Read and validate committed CSV inputs as session-local temporary views (daily only).
if mode == "daily":
    def repo_file(filename):
        # Same git-source cwd lookup as the deployed notebook.
        for candidate in (os.path.join(os.getcwd(), filename),
                          os.path.join(os.getcwd(), "notebooks", "funders", filename)):
            if os.path.exists(candidate):
                return candidate
        raise FileNotFoundError("notebooks/funders/" + filename)

    with open(repo_file("fundref_predecessor_bridge.csv"), newline="") as handle:
        bridge = list(csv.DictReader(handle))
    edges = sorted({(row["predecessor_fundref_id"], row["successor_fundref_id"]) for row in bridge})
    assert edges and all(re.fullmatch(r"[0-9]{5,}", value) for edge in edges for value in edge), "Invalid predecessor bridge CSV"
    spark.createDataFrame(edges, "old_id STRING, new_id STRING").createOrReplaceTempView("fr1006_predecessor_edges")

    with open(repo_file("edits.csv"), newline="") as handle:
        edits = [row for row in csv.DictReader(handle) if (row.get("action") or "").strip()]
    edit_rows = []
    for row_no, row in enumerate(edits):
        action = row["action"]
        assert action in ("merge", "withdraw", "set_ror", "clear_ror"), f"edits.csv: unknown action {action!r}"
        assert (row.get("reviewed_by") or "").strip(), f"edits.csv: row for F{row['funder_id']} has no reviewed_by"
        fid, target = int(row["funder_id"]), (row.get("target") or "").strip()
        ror_rid = None
        if action == "merge":
            target = str(int(target))
            assert int(target) != fid, f"edits.csv: F{fid} merged into itself"
        elif action == "set_ror":
            assert re.fullmatch(r"(https?://(www\.)?ror\.org/)?0[0-9a-z]{8}/?", target, re.I), f"edits.csv: bad ROR id {target!r} for F{fid}"
            ror_rid = target.rstrip("/")[-9:].lower()
        edit_rows.append((row_no, action, fid, target, ror_rid))
    spark.createDataFrame(edit_rows, "row_no BIGINT, action STRING, funder_id BIGINT, target STRING, ror_rid STRING").createOrReplaceTempView("fr1006_edits")


In [ ]:
%sql
-- Apply reviewed edits in CSV order, with each original precondition before its writes.
BEGIN
IF :mode = 'daily' THEN
FOR edit AS SELECT * FROM fr1006_edits ORDER BY row_no DO
IF edit.action = 'merge' THEN
-- edit_merge_check
SELECT assert_true((SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder') WHERE funder_id = CAST(edit.target AS BIGINT) AND status = 'active') = 1 AND ((SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder') WHERE funder_id = edit.funder_id AND status = 'active') = 1 OR (SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder')
  WHERE funder_id = edit.funder_id AND status = 'merged' AND merged_into_id = CAST(edit.target AS BIGINT)) = 1), concat('edits.csv merge F', CAST(edit.funder_id AS STRING), ' -> F', edit.target, ': target not active or loser not active')) ok;
-- edit_merge
MERGE INTO IDENTIFIER(trim(:prefix) || 'funder') f USING (SELECT edit.funder_id l, CAST(edit.target AS BIGINT) w) d ON f.funder_id = d.l OR f.merged_into_id = d.l
WHEN MATCHED AND f.funder_id <> d.w AND NOT (f.status = 'merged' AND f.merged_into_id = d.w) THEN UPDATE SET f.status = 'merged', f.merged_into_id = d.w,
  f.ror_id = CASE WHEN f.funder_id = d.l THEN NULL ELSE f.ror_id END, f.ror_primary = CASE WHEN f.funder_id = d.l THEN NULL ELSE f.ror_primary END,
  f.updated_date = current_timestamp(), f.run_id = ('fd' || trim(:run_id));
-- edit_merge_ids
UPDATE IDENTIFIER(trim(:prefix) || 'funder_id') SET funder_id = CAST(edit.target AS BIGINT), run_id = ('fd' || trim(:run_id)) WHERE funder_id = edit.funder_id AND id_type NOT IN ('name', 'ror_cleared');
ELSEIF edit.action = 'withdraw' THEN
-- edit_withdraw
UPDATE IDENTIFIER(trim(:prefix) || 'funder') SET status = 'withdrawn', ror_id = NULL, ror_primary = NULL, updated_date = current_timestamp(), run_id = ('fd' || trim(:run_id)) WHERE funder_id = edit.funder_id AND status <> 'withdrawn';
ELSEIF edit.action = 'set_ror' THEN
-- edit_set_ror_check
SELECT assert_true((SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder') WHERE funder_id = edit.funder_id AND status = 'active') = 1, concat('edits.csv set_ror: F', CAST(edit.funder_id AS STRING), ' is not active')) ok;
-- edit_set_ror
UPDATE IDENTIFIER(trim(:prefix) || 'funder') SET ror_id = ('https://ror.org/' || edit.ror_rid), ror_primary = true, updated_date = current_timestamp(), run_id = ('fd' || trim(:run_id)) WHERE funder_id = edit.funder_id AND (ror_id IS DISTINCT FROM ('https://ror.org/' || edit.ror_rid) OR NOT ror_primary);
-- edit_set_ror_demote
UPDATE IDENTIFIER(trim(:prefix) || 'funder') SET ror_primary = false, run_id = ('fd' || trim(:run_id)) WHERE ror_id = ('https://ror.org/' || edit.ror_rid) AND funder_id <> edit.funder_id AND ror_primary;
-- edit_set_ror_old
UPDATE IDENTIFIER(trim(:prefix) || 'funder_id') SET id_type = 'ror_cleared', id_value = id_value || concat(':', CAST(edit.funder_id AS STRING), ''), run_id = ('fd' || trim(:run_id)) WHERE funder_id = edit.funder_id AND id_type = 'ror' AND id_value <> edit.ror_rid;
-- edit_set_ror_id
MERGE INTO IDENTIFIER(trim(:prefix) || 'funder_id') i USING (SELECT edit.ror_rid v) d ON i.id_type = 'ror' AND i.id_value = d.v
WHEN MATCHED AND i.funder_id <> edit.funder_id THEN UPDATE SET i.funder_id = edit.funder_id, i.run_id = ('fd' || trim(:run_id))
WHEN NOT MATCHED THEN INSERT (id_type, id_value, funder_id, origin_funder_id, is_primary, source, added_at, run_id)
  VALUES ('ror', d.v, edit.funder_id, edit.funder_id, true, 'edit', current_timestamp(), ('fd' || trim(:run_id)));
ELSEIF edit.action = 'clear_ror' THEN
-- edit_clear_ror
UPDATE IDENTIFIER(trim(:prefix) || 'funder') SET ror_id = NULL, ror_primary = NULL, updated_date = current_timestamp(), run_id = ('fd' || trim(:run_id)) WHERE funder_id = edit.funder_id AND ror_id IS NOT NULL;
-- edit_clear_ror_id
UPDATE IDENTIFIER(trim(:prefix) || 'funder_id') SET id_type = 'ror_cleared', id_value = id_value || concat(':', CAST(edit.funder_id AS STRING), ''), run_id = ('fd' || trim(:run_id)) WHERE funder_id = edit.funder_id AND id_type = 'ror';
END IF;
END FOR;
END IF;
END;


In [ ]:
%sql
-- upsert_sources
BEGIN
IF :mode = 'daily' THEN
MERGE INTO IDENTIFIER(trim(:prefix) || 'funder_source_record') t
USING (
  WITH ror AS (
    SELECT lower(regexp_extract(id, '([0-9a-z]{9})/*$', 1)) rid, id ror_url, status, coalesce(array_contains(types, 'funder'), false) is_funder,
      nullif(trim(get(filter(names, n -> array_contains(n.types, 'ror_display')), 0).value), '') display_name,
      transform(coalesce(names, array()), n -> n.value) full_names,
      array_sort(array_distinct(filter(transform(flatten(transform(filter(coalesce(external_ids, array()), e -> lower(e.type) = 'fundref'),
        e -> coalesce(e.all, array()))), x -> regexp_extract(trim(x), '^(?:(?:https?://)?(?:dx\\.)?(?:doi\\.org/)?10\\.13039/)?([0-9]{5,})$', 1)), x -> x <> ''))) fundrefs,
      nullif(upper(trim(get(locations, 0).geonames_details.country_code)), '') country,
      get(filter(links, l -> l.type = 'website'), 0).value homepage,
      transform(filter(coalesce(relationships, array()), r -> r.type = 'successor'), r -> lower(regexp_extract(r.id, '([0-9a-z]{9})/*$', 1))) successors,
      to_json(named_struct('s', status, 't', types, 'n', names, 'e', external_ids, 'r', relationships, 'l', locations, 'k', links)) j
    FROM openalex.institutions.ror
  )
  , edges AS (SELECT * FROM fr1006_predecessor_edges),
  bridge AS (
    SELECT e.old_id, min(r.rid) rid FROM edges e JOIN (SELECT rid, explode(fundrefs) k FROM ror) r ON r.k = e.new_id
    LEFT ANTI JOIN (SELECT explode(fundrefs) k FROM ror) own ON own.k = e.old_id
    GROUP BY e.old_id HAVING count(DISTINCT r.rid) = 1
  ), xw AS (SELECT rid, array_sort(collect_set(old_id)) ids FROM bridge GROUP BY rid),
  cr_cited AS (
    SELECT regexp_extract(trim(f.doi), '^(?:(?:https?://)?(?:dx\\.)?(?:doi\\.org/)?10\\.13039/)?([0-9]{5,})$', 1) v, max(f.name) name, count(DISTINCT lm.work_id) works
    FROM openalex.works.locations_mapped lm LATERAL VIEW explode(lm.funders) AS f
    WHERE lm.provenance = 'crossref' AND lm.work_id IS NOT NULL AND regexp_extract(trim(f.doi), '^(?:(?:https?://)?(?:dx\\.)?(?:doi\\.org/)?10\\.13039/)?([0-9]{5,})$', 1) <> '' GROUP BY 1
  ), dc AS (   -- DataCite fundingReferences: every ROR / FundRef id DataCite records name as a funder, with how many DOIs cite it
    SELECT kt, v, count(DISTINCT native_id) dois, max(funder_name) name FROM (
      SELECT native_id, funder_name, CASE WHEN lower(declared_type) = 'ror' THEN 'ror' ELSE 'fundref' END kt,   -- the declared namespace decides
        CASE WHEN lower(declared_type) = 'ror' THEN lower(regexp_extract(identifier, '(0[0-9a-z]{8})/*$', 1)) ELSE regexp_extract(trim(identifier), '^(?:(?:https?://)?(?:dx\\.)?(?:doi\\.org/)?10\\.13039/)?([0-9]{5,})$', 1) END v
      FROM openalex.awards.datacite_work_funder_evidence
      WHERE (lower(declared_type) = 'crossref funder id' AND regexp_extract(trim(identifier), '^(?:(?:https?://)?(?:dx\\.)?(?:doi\\.org/)?10\\.13039/)?([0-9]{5,})$', 1) <> '')
         OR (lower(declared_type) = 'ror' AND trim(identifier) RLIKE '^((?i)https?://(www[.])?ror[.]org/)?0[0-9a-z]{8}/*$'))
    GROUP BY 1, 2
  )
  SELECT 'ror' source, r.rid source_key, display_name name, full_names alt_names, country, ror_url ror_id,
    array_sort(array_union(fundrefs, coalesce(xw.ids, array()))) fundref_ids,
    successors replaced_by, status record_status, is_funder, homepage, j record_json, sha2(concat(j, to_json(coalesce(xw.ids, array()))), 256) fingerprint
  FROM ror r LEFT JOIN xw ON xw.rid = r.rid
  WHERE is_funder OR size(fundrefs) > 0 OR r.rid IN (SELECT id_value FROM IDENTIFIER(trim(:prefix) || 'funder_id') WHERE id_type = 'ror')
     OR r.rid IN (SELECT v FROM dc WHERE kt = 'ror')
     OR r.rid IN (SELECT lower(regexp_extract(ror_id, '([0-9a-z]{9})/*$', 1)) FROM IDENTIFIER(trim(:prefix) || 'funder') WHERE ror_id IS NOT NULL)
  UNION ALL   -- DataCite names funders by id but has no funder list: cited ids are matching evidence only
  SELECT 'datacite_cited', kt || ':' || v, name, CAST(NULL AS ARRAY<STRING>), CAST(NULL AS STRING),
    CASE WHEN kt = 'ror' THEN 'https://ror.org/' || v END, CASE WHEN kt = 'fundref' THEN array(v) END, CAST(NULL AS ARRAY<STRING>),
    'active', true, CAST(NULL AS STRING),
    to_json(named_struct('dois', dois)), sha2(concat_ws('|', kt, v, name, CAST(dois AS STRING)), 256)
  FROM dc
  UNION ALL
  SELECT 'crossref_cited', 'fundref:' || v, name, CAST(NULL AS ARRAY<STRING>), CAST(NULL AS STRING),
    CAST(NULL AS STRING), array(v), CAST(NULL AS ARRAY<STRING>), 'active', false, CAST(NULL AS STRING),
    to_json(named_struct('works', works)), sha2(concat_ws('|', v, name, CAST(works AS STRING)), 256)
  FROM cr_cited
  UNION ALL
  SELECT 'crossref_crosswalk', old_id, CAST(NULL AS STRING), CAST(NULL AS ARRAY<STRING>), CAST(NULL AS STRING),
    'https://ror.org/' || rid, array(old_id), CAST(NULL AS ARRAY<STRING>), 'active', false, CAST(NULL AS STRING),
    to_json(named_struct('ror', rid)), sha2(rid, 256)
  FROM bridge
) s
ON t.source = s.source AND t.source_key = s.source_key
WHEN MATCHED AND (t.fingerprint <> s.fingerprint OR t.record_status = 'gone') THEN UPDATE SET
  t.name = s.name, t.alt_names = s.alt_names, t.country = s.country, t.ror_id = s.ror_id, t.fundref_ids = s.fundref_ids,
  t.replaced_by = s.replaced_by, t.record_status = s.record_status, t.is_funder = s.is_funder, t.homepage = s.homepage,
  t.record_json = s.record_json, t.fingerprint = s.fingerprint, t.fetched_at = current_timestamp(), t.changed_run_id = ('fd' || trim(:run_id))
WHEN NOT MATCHED THEN INSERT (source, source_key, name, alt_names, country, ror_id, fundref_ids, replaced_by, record_status, is_funder,
  homepage, record_json, fingerprint, first_seen_at, fetched_at, changed_run_id)
  VALUES (s.source, s.source_key, s.name, s.alt_names, s.country, s.ror_id, s.fundref_ids, s.replaced_by, s.record_status, s.is_funder,
  s.homepage, s.record_json, s.fingerprint, current_timestamp(), current_timestamp(), ('fd' || trim(:run_id)))
-- Legacy crossref_registry records retire to 'gone' once, then remain a no-op; keep their schema/history.
WHEN NOT MATCHED BY SOURCE AND t.source IN ('ror', 'crossref_registry', 'crossref_crosswalk', 'crossref_cited', 'datacite_cited') AND t.record_status <> 'gone' THEN UPDATE SET
  t.record_status = 'gone', t.changed_run_id = ('fd' || trim(:run_id));
END IF;
END;


In [ ]:
%sql
-- decide
BEGIN
IF :mode = 'daily' THEN
CREATE OR REPLACE TABLE IDENTIFIER(trim(:prefix) || 'funder_decision_tonight') AS
SELECT ('fd' || trim(:run_id)) run_id, current_timestamp() decided_at, * FROM (WITH
names AS (SELECT f.funder_id,
  array_distinct(filter(transform(coalesce(concat(array(f.display_name, seed.name), coalesce(f.alt_names, array()), coalesce(r.alt_names, array()), coalesce(a.names, array())), array()), x -> regexp_replace(trim(regexp_replace(lower(x), '[^\\p{L}\\p{N}]+', ' ')), '^the ', '')), x -> length(x) >= 4)) nk
FROM IDENTIFIER(trim(:prefix) || 'funder') f
LEFT JOIN IDENTIFIER(trim(:prefix) || 'funder_source_record') r ON r.source = 'ror' AND r.source_key = lower(regexp_extract(f.ror_id, '([0-9a-z]{9})/*$', 1)) AND r.record_status <> 'gone'
LEFT JOIN IDENTIFIER(trim(:prefix) || 'funder_source_record') seed ON seed.source = 'openalex_v41' AND seed.source_key = CAST(f.funder_id AS STRING)
LEFT JOIN (SELECT coalesce(f.merged_into_id, f.funder_id) funder_id, collect_set(a.name) names
  FROM openalex.common.funder_names_keep a JOIN IDENTIFIER(trim(:prefix) || 'funder') f ON f.funder_id = try_cast(regexp_extract(a.id, 'F([0-9]+)$', 1) AS BIGINT)
  WHERE upper(trim(a.keep)) = 'YES' GROUP BY 1) a ON a.funder_id = f.funder_id),
fa AS (SELECT f.funder_id, status, CASE WHEN ror_primary THEN lower(regexp_extract(ror_id, '([0-9a-z]{9})/*$', 1)) END rid, country_code, n.nk
       FROM IDENTIFIER(trim(:prefix) || 'funder') f JOIN names n ON n.funder_id = f.funder_id),
act AS (SELECT * FROM fa WHERE status = 'active'),
fid_fr AS (SELECT id_value k, funder_id FROM IDENTIFIER(trim(:prefix) || 'funder_id') WHERE id_type = 'fundref'),
fid_ror AS (SELECT id_value rid, funder_id FROM IDENTIFIER(trim(:prefix) || 'funder_id') WHERE id_type = 'ror'),
ror AS (SELECT r.source_key rid, r.ror_id ror_url, r.name, r.alt_names, r.country, r.fundref_ids fundrefs, r.is_funder, r.homepage,
          coalesce(from_json(get_json_object(r.record_json, '$.t'), 'array<string>'), array()) types,
          array_distinct(filter(transform(coalesce(concat(array(r.name), coalesce(r.alt_names, array())), array()), x -> regexp_replace(trim(regexp_replace(lower(x), '[^\\p{L}\\p{N}]+', ' ')), '^the ', '')), x -> length(x) >= 4)) nk
        FROM IDENTIFIER(trim(:prefix) || 'funder_source_record') r
        WHERE r.source = 'ror' AND r.record_status = 'active'),
fr_rors AS (SELECT k, array_sort(collect_set(rid)) rids FROM (SELECT rid, explode(fundrefs) k FROM ror) GROUP BY k),
allfr AS (SELECT * FROM fid_fr),
tinfo AS (SELECT funder_id, rid, nk, country_code FROM act),
owned AS (SELECT rid FROM fid_ror),
r1 AS (SELECT r.* FROM ror r LEFT ANTI JOIN owned o ON o.rid = r.rid),
r_t AS (   -- FundRef targets of each unowned ROR record, with name agreement and whether the target already has a ROR
  SELECT x.rid, x.k, a.funder_id, coalesce(arrays_overlap(x.nk, ti.nk), false) name_ok, ti.rid IS NOT NULL has_ror,
    size(coalesce(fr.rids, array())) n_rors
  FROM (SELECT rid, nk, explode(fundrefs) k FROM r1) x JOIN allfr a ON a.k = x.k JOIN tinfo ti ON ti.funder_id = a.funder_id
  LEFT JOIN fr_rors fr ON fr.k = x.k
  LEFT ANTI JOIN (SELECT split_part(id_value, ':', 1) rid, funder_id FROM IDENTIFIER(trim(:prefix) || 'funder_id') WHERE id_type = 'ror_cleared') rc   -- a person cleared it
    ON rc.rid = x.rid AND rc.funder_id = a.funder_id
),
r_agg AS (
  SELECT rid, array_sort(collect_set(funder_id)) tids, array_sort(collect_set(CASE WHEN name_ok THEN funder_id END)) cands,
    array_sort(collect_set(CASE WHEN name_ok AND NOT has_ror THEN funder_id END)) cands_free,
    max(CASE WHEN name_ok THEN n_rors END) cand_n_rors
  FROM r_t GROUP BY rid
),
r_w AS (SELECT x.rid, array_sort(collect_set(f.funder_id)) ids FROM (SELECT rid, explode(fundrefs) k FROM r1) x JOIN allfr a ON a.k = x.k
        JOIN IDENTIFIER(trim(:prefix) || 'funder') f ON f.funder_id = a.funder_id AND f.status = 'withdrawn' GROUP BY x.rid),
r_name AS (SELECT x.rid, array_sort(collect_set(a.funder_id)) ids FROM (SELECT rid, country, explode(nk) n FROM r1) x
           JOIN (SELECT funder_id, country_code, explode(nk) n FROM act WHERE rid IS NULL) a ON a.n = x.n AND a.country_code = x.country
           JOIN r1 rr ON rr.rid = x.rid AND a.funder_id <> abs(xxhash64(rr.ror_url)) % 9000000000
           GROUP BY x.rid),
r2 AS (
  SELECT r1.*, coalesce(g.tids, array()) tids, coalesce(g.cands, array()) cands, coalesce(g.cands_free, array()) cands_free,
    g.cand_n_rors, coalesce(rn.ids, array()) name_ids, coalesce(rw.ids, array()) withdrawn_ids, abs(xxhash64(r1.ror_url)) % 9000000000 mint_id,
    CASE
      WHEN size(coalesce(g.tids, array())) > 0 THEN CASE
        WHEN size(g.cands) = 0 THEN 'HOLD_NAME_DISAGREES'            -- 9 of 10 such links were programmes on the parent's record (09-28 grading)
        WHEN size(g.cands) > 1 THEN 'HOLD_FUNDREF_MULTI_FUNDER'
        WHEN size(g.cands_free) = 0 THEN 'HOLD_FUNDER_HAS_OTHER_ROR'
        WHEN g.cand_n_rors > 1 THEN 'HOLD_FUNDREF_MULTI_ROR'
        ELSE 'DEFER_ATTACH_ROR' END
      WHEN NOT r1.is_funder THEN 'IGNORE'
      WHEN size(coalesce(rw.ids, array())) > 0 THEN 'HOLD_FUNDREF_OF_WITHDRAWN'   -- its FundRef belongs to a funder deleted by hand
      WHEN size(coalesce(rn.ids, array())) > 0 THEN 'HOLD_NAME_ONLY_MATCH'
      WHEN r1.name IS NULL OR r1.country IS NULL THEN 'HOLD_MINT_INCOMPLETE'
      WHEN e.funder_id IS NOT NULL THEN 'HOLD_ID_COLLISION'
      ELSE 'MINT_ROR' END action
  FROM r1 LEFT JOIN r_agg g ON g.rid = r1.rid LEFT JOIN r_name rn ON rn.rid = r1.rid LEFT JOIN r_w rw ON rw.rid = r1.rid
  LEFT JOIN (SELECT funder_id, mint_key FROM IDENTIFIER(trim(:prefix) || 'funder')) e ON e.funder_id = abs(xxhash64(r1.ror_url)) % 9000000000
    AND e.mint_key IS DISTINCT FROM 'ror:' || r1.rid
),
r_multi AS (SELECT get(cands_free, 0) f FROM r2 WHERE action = 'ATTACH_ROR' GROUP BY 1 HAVING count(*) > 1),
rfin AS (SELECT r2.* EXCEPT (action), CASE WHEN action = 'ATTACH_ROR' AND m.f IS NOT NULL THEN 'HOLD_FUNDER_MULTI_ROR' ELSE action END action
         FROM r2 LEFT JOIN r_multi m ON m.f = get(r2.cands_free, 0))
SELECT 'ror' source, rid source_key, action,
  CASE WHEN action = 'MINT_ROR' THEN mint_id WHEN action IN ('ATTACH_ROR', 'DEFER_ATTACH_ROR') THEN get(cands_free, 0) WHEN action = 'HOLD_ID_COLLISION' THEN mint_id
       WHEN action IN ('HOLD_FUNDER_HAS_OTHER_ROR', 'HOLD_FUNDREF_MULTI_ROR', 'HOLD_FUNDER_MULTI_ROR') THEN get(cands, 0)
       WHEN action = 'HOLD_NAME_DISAGREES' THEN get(tids, 0) WHEN action = 'HOLD_NAME_ONLY_MATCH' THEN get(name_ids, 0)
       WHEN action = 'HOLD_FUNDREF_OF_WITHDRAWN' THEN get(withdrawn_ids, 0) END funder_id,
  rid ror_rid, ror_url, CAST(NULL AS STRING) fundref, name display_name, alt_names, country, homepage,
  to_json(named_struct('fundrefs', fundrefs, 'fundref_targets', tids, 'name_agreeing', cands, 'name_agreeing_without_ror', cands_free,
    'cand_fundref_rors', cand_n_rors, 'name_matches', name_ids, 'withdrawn_targets', withdrawn_ids, 'is_funder', is_funder, 'types', types)) evidence
FROM rfin WHERE action <> 'IGNORE');
END IF;
END;


In [ ]:
%sql
-- review_mints
BEGIN
IF :mode = 'daily' THEN
UPDATE IDENTIFIER(trim(:prefix) || 'funder_decision_tonight') SET action = CASE action WHEN 'MINT' THEN 'IGNORE_MINT_OFF' WHEN 'MINT_WITH_ROR' THEN 'IGNORE_MINT_OFF' WHEN 'MINT_ROR_DATACITE' THEN 'IGNORE_MINT_OFF' END WHERE action IN ('MINT', 'MINT_WITH_ROR', 'MINT_ROR_DATACITE');
END IF;
END;


In [ ]:
%sql
-- hold_batch_collisions
BEGIN
IF :mode = 'daily' THEN
UPDATE IDENTIFIER(trim(:prefix) || 'funder_decision_tonight') SET action = 'HOLD_ID_COLLISION'
WHERE action LIKE 'MINT%' AND funder_id IN (SELECT funder_id FROM IDENTIFIER(trim(:prefix) || 'funder_decision_tonight') WHERE action LIKE 'MINT%' GROUP BY 1 HAVING count(*) > 1);
END IF;
END;


In [ ]:
%sql
-- mint_fuse
BEGIN
IF :mode = 'daily' THEN
SELECT assert_true((SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder_decision_tonight') WHERE action LIKE 'MINT%') <= coalesce(nullif(CAST(:max_mints AS INT), 0), 200),
  concat('more than ', coalesce(nullif(CAST(:max_mints AS INT), 0), 200), ' mints tonight: stop for a person (fuse); nothing applied')) ok;
END IF;
END;


In [ ]:
%sql
-- never_issued
BEGIN
IF :mode = 'daily' THEN
SELECT assert_true((SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder_decision_tonight') d JOIN IDENTIFIER(trim(:prefix) || 'f_ids_ever_issued') e ON e.funder_id = d.funder_id
  LEFT ANTI JOIN IDENTIFIER(trim(:prefix) || 'funder') f ON f.funder_id = d.funder_id AND f.mint_key = CASE WHEN d.source = 'ror' THEN 'ror:' ELSE 'fundref:' END || d.source_key
  WHERE d.action LIKE 'MINT%') = 0, 'a mint candidate F id was issued before (ever-issued table); nothing applied (fail closed)') ok;
END IF;
END;


In [ ]:
%sql
-- log_decisions
BEGIN
IF :mode = 'daily' THEN
INSERT INTO IDENTIFIER(trim(:prefix) || 'funder_decision') SELECT run_id, decided_at, source, source_key, action, funder_id, ror_rid, fundref,
  display_name, country, evidence FROM IDENTIFIER(trim(:prefix) || 'funder_decision_tonight');
END IF;
END;


In [ ]:
%sql
-- mint_funders
BEGIN
IF :mode = 'daily' THEN
MERGE INTO IDENTIFIER(trim(:prefix) || 'funder') f USING (SELECT * FROM IDENTIFIER(trim(:prefix) || 'funder_decision_tonight') WHERE action LIKE 'MINT%') d ON f.funder_id = d.funder_id
WHEN NOT MATCHED THEN INSERT (funder_id, status, ror_id, ror_primary, display_name, alt_names, country_code, homepage_url, created_date, updated_date,
  created_by, run_id, mint_key)
VALUES (d.funder_id, 'active', d.ror_url, CASE WHEN d.ror_url IS NOT NULL THEN true END, d.display_name, d.alt_names, d.country, d.homepage, current_timestamp(), current_timestamp(),
  'mint_ror', ('fd' || trim(:run_id)),
  CASE WHEN d.source = 'ror' THEN 'ror:' ELSE 'fundref:' END || d.source_key);
END IF;
END;


In [ ]:
%sql
-- issue_ids
BEGIN
IF :mode = 'daily' THEN
INSERT INTO IDENTIFIER(trim(:prefix) || 'f_ids_ever_issued') SELECT f.funder_id, f.created_by, f.created_date FROM IDENTIFIER(trim(:prefix) || 'funder') f
JOIN (SELECT DISTINCT funder_id FROM IDENTIFIER(trim(:prefix) || 'funder_decision_tonight') WHERE action LIKE 'MINT%') d ON d.funder_id = f.funder_id
LEFT ANTI JOIN IDENTIFIER(trim(:prefix) || 'f_ids_ever_issued') e ON e.funder_id = f.funder_id;
END IF;
END;


In [ ]:
%sql
-- attach_ror
BEGIN
IF :mode = 'daily' THEN
MERGE INTO IDENTIFIER(trim(:prefix) || 'funder') f USING (SELECT * FROM IDENTIFIER(trim(:prefix) || 'funder_decision_tonight') WHERE action = 'ATTACH_ROR') d ON f.funder_id = d.funder_id
WHEN MATCHED AND (f.ror_id IS NULL OR NOT f.ror_primary) AND f.status = 'active'   -- a secondary (shared) ROR gives way to the funder's own
  THEN UPDATE SET f.ror_id = d.ror_url, f.ror_primary = true, f.updated_date = current_timestamp(), f.run_id = ('fd' || trim(:run_id));
END IF;
END;


In [ ]:
%sql
-- sync_fields
BEGIN
IF :mode = 'daily' THEN
SELECT 0 ok;
END IF;
END;


In [ ]:
%sql
-- withdraw_unlinked
BEGIN
IF :mode = 'daily' THEN
SELECT 0 ok;
END IF;
END;


In [ ]:
%sql
-- add_ids
BEGIN
IF :mode = 'daily' THEN
MERGE INTO IDENTIFIER(trim(:prefix) || 'funder_id') i USING (
  SELECT 'ror' id_type, ror_rid id_value, funder_id, funder_id origin, true prim, lower(action) src
  FROM IDENTIFIER(trim(:prefix) || 'funder_decision_tonight') WHERE action IN ('MINT_ROR', 'ATTACH_ROR')
  UNION ALL
  SELECT 'fundref', x.k, min(i.funder_id), min(i.funder_id), false, 'ror_crosswalk'
  FROM (SELECT source_key, explode(fundref_ids) k FROM IDENTIFIER(trim(:prefix) || 'funder_source_record') WHERE source = 'ror' AND record_status = 'active') x
  JOIN (SELECT id_value, funder_id FROM IDENTIFIER(trim(:prefix) || 'funder_id') WHERE id_type = 'ror' UNION SELECT ror_rid, funder_id FROM IDENTIFIER(trim(:prefix) || 'funder_decision_tonight') WHERE action IN ('MINT_ROR', 'ATTACH_ROR')) i ON i.id_value = x.source_key
  WHERE true AND x.k NOT IN (
    SELECT k FROM (SELECT source_key, explode(fundref_ids) k FROM IDENTIFIER(trim(:prefix) || 'funder_source_record') WHERE source = 'ror' AND record_status = 'active')
    GROUP BY k HAVING count(DISTINCT source_key) > 1)
  GROUP BY x.k HAVING count(DISTINCT x.source_key) = 1
) d ON i.id_type = d.id_type AND i.id_value = d.id_value
WHEN NOT MATCHED THEN INSERT (id_type, id_value, funder_id, origin_funder_id, is_primary, source, added_at, run_id)
VALUES (d.id_type, d.id_value, d.funder_id, d.origin, d.prim, d.src, current_timestamp(), ('fd' || trim(:run_id)));
END IF;
END;


In [ ]:
%sql
-- refresh_names
BEGIN
IF :mode = 'daily' THEN
MERGE INTO IDENTIFIER(trim(:prefix) || 'funder_id') i USING (   -- name keys owned by exactly one active funder; rebuilt every run
  SELECT n id_value, min(funder_id) funder_id FROM (
    SELECT f.funder_id, explode(n.nk) n FROM IDENTIFIER(trim(:prefix) || 'funder') f JOIN (SELECT f.funder_id,
  array_distinct(filter(transform(coalesce(concat(array(f.display_name, seed.name), coalesce(f.alt_names, array()), coalesce(r.alt_names, array()), coalesce(a.names, array())), array()), x -> regexp_replace(trim(regexp_replace(lower(x), '[^\\p{L}\\p{N}]+', ' ')), '^the ', '')), x -> length(x) >= 4)) nk
FROM IDENTIFIER(trim(:prefix) || 'funder') f
LEFT JOIN IDENTIFIER(trim(:prefix) || 'funder_source_record') r ON r.source = 'ror' AND r.source_key = lower(regexp_extract(f.ror_id, '([0-9a-z]{9})/*$', 1)) AND r.record_status <> 'gone'
LEFT JOIN IDENTIFIER(trim(:prefix) || 'funder_source_record') seed ON seed.source = 'openalex_v41' AND seed.source_key = CAST(f.funder_id AS STRING)
LEFT JOIN (SELECT coalesce(f.merged_into_id, f.funder_id) funder_id, collect_set(a.name) names
  FROM openalex.common.funder_names_keep a JOIN IDENTIFIER(trim(:prefix) || 'funder') f ON f.funder_id = try_cast(regexp_extract(a.id, 'F([0-9]+)$', 1) AS BIGINT)
  WHERE upper(trim(a.keep)) = 'YES' GROUP BY 1) a ON a.funder_id = f.funder_id) n ON n.funder_id = f.funder_id WHERE f.status = 'active')
  GROUP BY n HAVING count(DISTINCT funder_id) = 1
) d ON i.id_type = 'name' AND i.id_value = d.id_value
WHEN MATCHED AND i.funder_id <> d.funder_id THEN UPDATE SET i.funder_id = d.funder_id, i.origin_funder_id = d.funder_id, i.run_id = ('fd' || trim(:run_id))
WHEN NOT MATCHED THEN INSERT (id_type, id_value, funder_id, origin_funder_id, is_primary, source, added_at, run_id)
  VALUES ('name', d.id_value, d.funder_id, d.funder_id, false, 'name_unique', current_timestamp(), ('fd' || trim(:run_id)))
WHEN NOT MATCHED BY SOURCE AND i.id_type = 'name' THEN DELETE;
END IF;
END;


In [ ]:
%sql
-- source_outcomes
BEGIN
IF :mode = 'daily' THEN
MERGE INTO IDENTIFIER(trim(:prefix) || 'funder_source_record') s USING (
  SELECT source, source_key, CASE WHEN action LIKE 'HOLD%' OR action LIKE 'REVIEW%' THEN 'held' WHEN action LIKE 'DEFER%' THEN 'deferred' WHEN action LIKE 'IGNORE%' THEN 'unmatched' ELSE 'accepted' END outcome,
    CASE WHEN action LIKE 'MINT%' OR action = 'ATTACH_ROR' THEN 'MATCH' ELSE action END reason, funder_id FROM IDENTIFIER(trim(:prefix) || 'funder_decision_tonight')
  UNION ALL   -- records already resolved by an id we hold
  SELECT s.source, s.source_key, 'accepted', 'MATCH', i.funder_id FROM IDENTIFIER(trim(:prefix) || 'funder_source_record') s
  JOIN IDENTIFIER(trim(:prefix) || 'funder_id') i ON i.id_type = CASE s.source WHEN 'ror' THEN 'ror' ELSE 'fundref' END AND i.id_value = s.source_key
  LEFT ANTI JOIN IDENTIFIER(trim(:prefix) || 'funder_decision_tonight') n ON n.source = s.source AND n.source_key = s.source_key
  WHERE s.source IN ('ror', 'crossref_crosswalk')
  UNION ALL   -- DataCite-cited ids: accepted once a funder holds the id, else unmatched evidence, never a mint review
  SELECT s.source, s.source_key, CASE WHEN coalesce(i.funder_id, n.funder_id) IS NOT NULL THEN 'accepted' ELSE 'unmatched' END,
    CASE WHEN i.funder_id IS NOT NULL THEN 'MATCH' WHEN n.funder_id IS NOT NULL THEN 'MATCH_NAME' ELSE 'CITED_ID_UNMATCHED_MINT_OFF' END, coalesce(i.funder_id, n.funder_id)
  FROM IDENTIFIER(trim(:prefix) || 'funder_source_record') s LEFT JOIN IDENTIFIER(trim(:prefix) || 'funder_id') i ON i.id_type = split_part(s.source_key, ':', 1) AND i.id_value = split_part(s.source_key, ':', 2)
  LEFT JOIN IDENTIFIER(trim(:prefix) || 'funder_id') n ON n.id_type = 'name' AND n.id_value = regexp_replace(trim(regexp_replace(lower(s.name), '[^\\p{L}\\p{N}]+', ' ')), '^the ', '')
  WHERE s.source IN ('datacite_cited', 'crossref_cited') AND s.record_status <> 'gone'
) d ON s.source = d.source AND s.source_key = d.source_key
WHEN MATCHED AND (s.outcome IS DISTINCT FROM d.outcome OR s.reason IS DISTINCT FROM d.reason OR s.funder_id IS DISTINCT FROM d.funder_id)
  THEN UPDATE SET s.outcome = d.outcome, s.reason = d.reason, s.funder_id = d.funder_id, s.decided_run_id = ('fd' || trim(:run_id));
END IF;
END;


In [ ]:
%sql
-- checks
BEGIN
IF :mode = 'daily' THEN
SELECT
  assert_true((SELECT count(*) FROM (SELECT ror_id FROM IDENTIFIER(trim(:prefix) || 'funder') WHERE status = 'active' AND ror_primary GROUP BY 1 HAVING count(*) > 1)) = 0,
    'ROR id primary on two active funders') c_ror_unique,
  assert_true((SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder') m LEFT JOIN IDENTIFIER(trim(:prefix) || 'funder') w ON w.funder_id = m.merged_into_id
    WHERE m.status = 'merged' AND (w.funder_id IS NULL OR w.status <> 'active')) = 0, 'merged funder not pointing at an active funder') c_merge_target,
  assert_true((SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder_id') i LEFT ANTI JOIN IDENTIFIER(trim(:prefix) || 'funder') f ON f.funder_id = i.funder_id) = 0, 'funder_id points at a missing funder') c_id_target,
  assert_true((SELECT count(*) FROM (SELECT id_type, id_value FROM IDENTIFIER(trim(:prefix) || 'funder_id') GROUP BY 1, 2 HAVING count(*) > 1)) = 0, 'identifier mapped twice') c_id_unique,
  assert_true((SELECT count(*) FROM (SELECT funder_id FROM IDENTIFIER(trim(:prefix) || 'funder') GROUP BY 1 HAVING count(*) > 1)) = 0, 'F id issued twice') c_f_unique,
  assert_true((SELECT count(*) FROM (SELECT funder_id FROM fr1006_prev_funder EXCEPT SELECT funder_id FROM IDENTIFIER(trim(:prefix) || 'funder'))) = 0,
    'an issued F id disappeared') c_never_deleted,
  assert_true((SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder') f LEFT ANTI JOIN IDENTIFIER(trim(:prefix) || 'funder_id') i ON i.id_type = 'ror' AND i.id_value = lower(regexp_extract(f.ror_id, '([0-9a-z]{9})/*$', 1)) AND i.funder_id = f.funder_id
    WHERE f.status = 'active' AND f.ror_primary) = 0, 'primary ror_id without its funder_id row') c_ror_indexed;
END IF;
END;


In [ ]:
%sql
-- publish_compat
BEGIN
IF :mode = 'daily' THEN
CREATE OR REPLACE TABLE IDENTIFIER(trim(:prefix) || 'funders_compat') AS
SELECT f.funder_id, CAST(get_json_object(v41.j, '$.crossref_id') AS BIGINT) crossref_id,
  get_json_object(v41.j, '$.location') location, f.display_name,
  CASE WHEN f.alt_names <=> from_json(get_json_object(v41.j, '$.alternate_titles'), 'array<string>') THEN get_json_object(v41.j, '$.alternate_titles') ELSE to_json(coalesce(f.alt_names, array())) END alternate_titles,
  get_json_object(v41.j, '$.uri') uri,
  coalesce(get_json_object(v41.j, '$.replaces'), CASE WHEN v41.j IS NULL THEN '[]' END) replaces,
  coalesce(get_json_object(v41.j, '$.replaced_by'), CASE WHEN v41.j IS NULL THEN '[]' END) replaced_by,
  get_json_object(v41.j, '$.tokens') tokens, f.ror_id,
  CASE WHEN f.status = 'merged' THEN f.merged_into_id END merge_into_id,
  CASE WHEN f.status = 'merged' THEN coalesce(CASE WHEN get_json_object(v41.j, '$.merge_into_id') = CAST(f.merged_into_id AS STRING) THEN CAST(get_json_object(v41.j, '$.merge_into_date') AS TIMESTAMP) END, f.updated_date) END merge_into_date,
  coalesce(get_json_object(v41.j, '$.created_date'), CAST(f.created_date AS STRING)) created_date, f.wikidata_id,
  get_json_object(v41.j, '$.doi') doi,
  f.country_code, f.homepage_url, f.description, f.image_url, f.image_thumbnail_url, get_json_object(v41.j, '$.json_entity_hash') json_entity_hash, f.updated_date
FROM IDENTIFIER(trim(:prefix) || 'funder') f
LEFT JOIN (SELECT CAST(source_key AS BIGINT) funder_id, record_json j FROM IDENTIFIER(trim(:prefix) || 'funder_source_record') WHERE source = 'openalex_v41') v41 ON v41.funder_id = f.funder_id
LEFT JOIN IDENTIFIER(trim(:prefix) || 'funder_source_record') r ON r.source = 'ror' AND r.source_key = lower(regexp_extract(f.ror_id, '([0-9a-z]{9})/*$', 1)) AND r.record_status <> 'gone'
WHERE f.status IN ('active', 'merged');
END IF;
END;


In [ ]:
%sql
-- Seed funders_api only if absent, preserving served updated_date on the first daily run.
BEGIN
IF :mode = 'daily' THEN
CREATE TABLE IF NOT EXISTS IDENTIFIER(trim(:prefix) || 'funders_api') AS SELECT * FROM openalex.funders.funders_api;
END IF;
END;


In [ ]:
%sql
-- funders_api_0
BEGIN
IF :mode = 'daily' THEN
-- Snapshot funders_api hashes before rebuild
CREATE OR REPLACE TABLE IDENTIFIER(trim(:prefix) || 'funders_api_hash') AS
SELECT id, updated_date,
  xxhash64(CONCAT_WS('|',
    CAST(id AS STRING),
    COALESCE(display_name, ''),
    COALESCE(TO_JSON(alternate_titles), '[]'),
    COALESCE(country_code, ''),
    COALESCE(description, ''),
    COALESCE(homepage_url, ''),
    COALESCE(image_url, ''),
    COALESCE(image_thumbnail_url, ''),
    COALESCE(TO_JSON(ids), '{}'),
    COALESCE(CAST(works_count AS STRING), '0'),
    COALESCE(CAST(cited_by_count AS STRING), '0'),
    COALESCE(CAST(awards_count AS STRING), '0'),
    COALESCE(TO_JSON(summary_stats), '{}'),
    COALESCE(TO_JSON(roles), '[]'),
    COALESCE(TO_JSON(counts_by_year), '[]')
  )) AS content_hash
FROM IDENTIFIER(trim(:prefix) || 'funders_api');
END IF;
END;


In [ ]:
%sql
-- funders_api_1
BEGIN
IF :mode = 'daily' THEN
CREATE OR REPLACE TABLE IDENTIFIER(trim(:prefix) || 'funders_api') AS
WITH
-- Base from funders array (for work-funder relationships)
wf_base AS (
  SELECT
    CAST(REPLACE(funder.id, 'https://openalex.org/F', '') AS BIGINT) AS funder_id,
    CAST(w.id AS BIGINT) AS work_id
  FROM openalex.works.openalex_works w
  LATERAL VIEW EXPLODE(w.funders) AS funder
  WHERE funder.id IS NOT NULL 
    AND w.id IS NOT NULL
    AND funder.id LIKE 'https://openalex.org/F%'
),
-- Enrich with work-level signals needed for metrics
wf_enriched AS (
  SELECT
    b.funder_id,
    b.work_id,
    COALESCE(w.publication_year, YEAR(w.publication_date)) AS pub_year,
    CAST(w.cited_by_count AS INT) AS cited_by_count,
    CAST(w.open_access.is_oa AS BOOLEAN) AS is_oa
  FROM wf_base b
  JOIN openalex.works.openalex_works w
    ON w.id = b.work_id
),
funder_unique_awards AS (
  SELECT
    funder_id,
    COUNT(*) AS unique_awards_count
  FROM openalex.awards.openalex_awards
  WHERE funder_id IS NOT NULL
  GROUP BY funder_id
),
-- Per-funder metrics (works, citations, OA, grants; plus helpers for summary_stats)
funder_metrics AS (
  SELECT
    funder_id,
    /* counts over unique works */
    CAST(COUNT(DISTINCT work_id) AS INT) AS works_count,
    CAST(SUM(cited_by_count) AS INT)     AS cited_by_count,
    CAST(SUM(CASE WHEN is_oa THEN 1 ELSE 0 END) AS INT) AS oa_works_count,
    /* summary_stats helpers */
    CAST(AVG(CASE WHEN pub_year >= YEAR(current_date()) - 2 THEN cited_by_count END) AS DOUBLE) AS two_year_mean,
    CAST(COUNT_IF(cited_by_count >= 10) AS INT) AS i10_index,
    SORT_ARRAY(
      TRANSFORM(
        FILTER(COLLECT_LIST(cited_by_count), x -> x IS NOT NULL),
        x -> CAST(x AS INT)
      ),
      false
    ) AS sorted_citations
  FROM wf_enriched
  GROUP BY funder_id
),
-- Funder Roles CTE
funder_roles AS (
  WITH entity_links_expanded AS (
    SELECT 
      f.funder_id AS funder_id,
      el.id_1,
      el.id_2
    FROM IDENTIFIER(trim(:prefix) || 'funders_compat') f
    INNER JOIN openalex.mid.entity_link el 
      ON (el.id_1 = CONCAT('F', f.funder_id) OR el.id_2 = CONCAT('F', f.funder_id))
  ),
  all_entity_ids AS (
    -- Funder's own role
    SELECT 
      f.funder_id AS funder_id,
      CONCAT('F', f.funder_id) AS entity_id,
      'funder' AS role
    FROM IDENTIFIER(trim(:prefix) || 'funders_compat') f
    
    UNION ALL
    
    -- Linked institution/publisher roles
    SELECT 
      funder_id,
      CASE 
        WHEN id_1 LIKE 'F%' THEN id_2 
        ELSE id_1 
      END AS entity_id,
      CASE 
        WHEN id_1 LIKE 'I%' OR id_2 LIKE 'I%' THEN 'institution'
        WHEN id_1 LIKE 'P%' OR id_2 LIKE 'P%' THEN 'publisher'
      END AS role
    FROM entity_links_expanded
    WHERE (id_1 LIKE 'I%' OR id_1 LIKE 'P%' OR id_2 LIKE 'I%' OR id_2 LIKE 'P%')
  ),
  roles_with_counts AS (
    SELECT 
      ae.funder_id,
      ae.role,
      CONCAT('https://openalex.org/', ae.entity_id) AS id,
      CASE 
        -- For funder: use works_count from funder_metrics
        WHEN ae.role = 'funder' THEN fm.works_count
        -- For institution: join to institutions_api table
        WHEN ae.role = 'institution' THEN i_api.works_count
        -- For publisher: join to publishers_api table
        WHEN ae.role = 'publisher' THEN p_api.works_count
      END AS works_count
    FROM all_entity_ids ae
    -- Join to funder_metrics for funder works_count
    LEFT JOIN funder_metrics fm 
      ON ae.role = 'funder' AND ae.funder_id = fm.funder_id
    -- Join to institutions_api for institution works_count
    LEFT JOIN openalex.institutions.institutions_api i_api
      ON ae.role = 'institution' AND ae.entity_id = CONCAT('I', i_api.id)
    -- Join to publishers_api for publisher works_count
    LEFT JOIN openalex.publishers.publishers_api p_api
      ON ae.role = 'publisher' AND ae.entity_id = CONCAT('P', p_api.id)
  ),
  deduplicated_roles AS (
    SELECT 
      funder_id,
      role,
      id,
      CAST(COALESCE(works_count, 0) AS INT) AS works_count,
      ROW_NUMBER() OVER (
        PARTITION BY funder_id, role 
        ORDER BY works_count DESC, id
      ) AS rn
    FROM roles_with_counts
  )
  SELECT 
    funder_id,
    COLLECT_LIST(
      STRUCT(role, id, works_count)
    ) AS roles
  FROM deduplicated_roles
  WHERE rn = 1  -- Keep only one per role type
  GROUP BY funder_id
)

SELECT
  f.funder_id AS id,
  f.display_name,
  from_json(f.alternate_titles, 'ARRAY<STRING>') AS alternate_titles,
  f.country_code,
  f.description,
  f.homepage_url,
  f.image_url,
  f.image_thumbnail_url,
  STRUCT(
    CONCAT('https://openalex.org/F', f.funder_id) AS openalex,
    f.ror_id       AS ror,
    f.wikidata_id  AS wikidata,
    f.crossref_id  AS crossref,
    f.doi          AS doi
  ) AS ids,

  /* totals */
  COALESCE(m.works_count, 0)      AS works_count,
  COALESCE(m.cited_by_count, 0)   AS cited_by_count,
  COALESCE(fua.unique_awards_count, 0) AS awards_count,

  -- Roles
  COALESCE(fr.roles, ARRAY()) AS roles,

  /* counts_by_year */
  (
    SELECT
      SORT_ARRAY(
        COLLECT_LIST(
          STRUCT(
            year,
            works_count,
            oa_works_count,
            cited_by_count
          )
        ),
        false
      )
    FROM (
      SELECT
        CAST(e.pub_year AS INT) AS year,
        CAST(COUNT(DISTINCT e.work_id) AS INT) AS works_count,
        CAST(SUM(CASE WHEN e.is_oa THEN 1 ELSE 0 END) AS INT) AS oa_works_count,
        CAST(SUM(e.cited_by_count) AS INT) AS cited_by_count
      FROM wf_enriched e
      WHERE e.funder_id = f.funder_id
        AND e.pub_year IS NOT NULL
      GROUP BY CAST(e.pub_year AS INT)
    )
  ) AS counts_by_year,

  /* summary_stats */
  NAMED_STRUCT(
    '2yr_mean_citedness', COALESCE(m.two_year_mean, 0.0),
    'h_index',
      CAST(
        ARRAY_MAX(
          ZIP_WITH(
            m.sorted_citations,
            SEQUENCE(1, SIZE(m.sorted_citations)),
            (citation, rank) -> IF(citation >= rank, rank, 0)
          )
        ) AS INT
      ),
    'i10_index', COALESCE(m.i10_index, 0)
  ) AS summary_stats,

  /* dates */
  DATE_TRUNC('SECOND', CAST(f.created_date AS TIMESTAMP)) AS created_date,
  CAST(NULL AS TIMESTAMP) AS updated_date

FROM IDENTIFIER(trim(:prefix) || 'funders_compat') f
LEFT JOIN funder_metrics m
  ON m.funder_id = f.funder_id
LEFT JOIN funder_unique_awards fua
  ON f.funder_id = fua.funder_id
LEFT JOIN funder_roles fr ON f.funder_id = fr.funder_id
-- merge losers (MergeFunders.ipynb alias rows) stay in IDENTIFIER(trim(:prefix) || 'funders_compat') so DOI/name
-- matching keeps resolving, but must not surface as API entities
WHERE f.merge_into_id IS NULL;
END IF;
END;


In [ ]:
%sql
-- funders_api_2
BEGIN
IF :mode = 'daily' THEN
-- Funders: compare hashes and set updated_date only when content changed
WITH new_hashes AS (
  SELECT id,
    xxhash64(CONCAT_WS('|',
      CAST(id AS STRING),
      COALESCE(display_name, ''),
      COALESCE(TO_JSON(alternate_titles), '[]'),
      COALESCE(country_code, ''),
      COALESCE(description, ''),
      COALESCE(homepage_url, ''),
      COALESCE(image_url, ''),
      COALESCE(image_thumbnail_url, ''),
      COALESCE(TO_JSON(ids), '{}'),
      COALESCE(CAST(works_count AS STRING), '0'),
      COALESCE(CAST(cited_by_count AS STRING), '0'),
      COALESCE(CAST(awards_count AS STRING), '0'),
      COALESCE(TO_JSON(summary_stats), '{}'),
      COALESCE(TO_JSON(roles), '[]'),
      COALESCE(TO_JSON(counts_by_year), '[]')
    )) AS content_hash
  FROM IDENTIFIER(trim(:prefix) || 'funders_api')
)
MERGE INTO IDENTIFIER(trim(:prefix) || 'funders_api') AS target
USING (
  SELECT n.id,
    CASE
      WHEN p.id IS NULL THEN DATE_TRUNC('SECOND', CURRENT_TIMESTAMP())
      WHEN n.content_hash <> p.content_hash THEN DATE_TRUNC('SECOND', CURRENT_TIMESTAMP())
      ELSE p.updated_date
    END AS new_updated_date
  FROM new_hashes n
  LEFT JOIN IDENTIFIER(trim(:prefix) || 'funders_api_hash') p ON n.id = p.id
) AS source
ON target.id = source.id
WHEN MATCHED THEN UPDATE SET target.updated_date = source.new_updated_date;
END IF;
END;


In [ ]:
%sql
-- Print the original nightly diff report.
BEGIN
IF :mode = 'daily' THEN
SELECT 'source records changed' item, source detail, count(*) n FROM IDENTIFIER(trim(:prefix) || 'funder_source_record') WHERE changed_run_id = ('fd' || trim(:run_id)) GROUP BY source
UNION ALL SELECT 'decision', action, count(*) FROM IDENTIFIER(trim(:prefix) || 'funder_decision') WHERE run_id = ('fd' || trim(:run_id)) GROUP BY action
UNION ALL SELECT 'decision new since last run', d.action, count(*) FROM IDENTIFIER(trim(:prefix) || 'funder_decision') d
  LEFT ANTI JOIN IDENTIFIER(trim(:prefix) || 'funder_decision') o ON o.run_id <> ('fd' || trim(:run_id)) AND o.source = d.source AND o.source_key = d.source_key AND o.action = d.action
  WHERE d.run_id = ('fd' || trim(:run_id)) GROUP BY d.action
UNION ALL SELECT 'funders created', created_by, count(*) FROM IDENTIFIER(trim(:prefix) || 'funder') WHERE run_id = ('fd' || trim(:run_id)) AND created_by LIKE 'mint%'
  AND mint_key IN (SELECT CASE WHEN source = 'ror' THEN 'ror:' ELSE 'fundref:' END || source_key FROM IDENTIFIER(trim(:prefix) || 'funder_decision') WHERE run_id = ('fd' || trim(:run_id)) AND action LIKE 'MINT%')
  GROUP BY created_by
UNION ALL SELECT 'funders changed (now)', status, count(*) FROM IDENTIFIER(trim(:prefix) || 'funder') WHERE run_id = ('fd' || trim(:run_id)) AND NOT (created_by LIKE 'mint%' AND mint_key IN (
  SELECT CASE WHEN source = 'ror' THEN 'ror:' ELSE 'fundref:' END || source_key FROM IDENTIFIER(trim(:prefix) || 'funder_decision') WHERE run_id = ('fd' || trim(:run_id)) AND action LIKE 'MINT%')) GROUP BY status
UNION ALL SELECT 'ids added or repointed', id_type, count(*) FROM IDENTIFIER(trim(:prefix) || 'funder_id') WHERE run_id = ('fd' || trim(:run_id)) GROUP BY id_type
UNION ALL SELECT 'registry now', status, count(*) FROM IDENTIFIER(trim(:prefix) || 'funder') GROUP BY status
ORDER BY 1, 2;
END IF;
END;
